# Mixture Score Field

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

Gaussian-mixture score field bending between two modes.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/mixture-score/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch
from torch.distributions import Categorical, MixtureSameFamily, MultivariateNormal

# The demo's defaults: two isotropic Gaussians at (-gap/2, 0) and (gap/2, 0), weight pi, std s
gap, pi, s = 2.4, 0.5, 0.75
means = torch.tensor([[-gap / 2, 0.0], [gap / 2, 0.0]], dtype=torch.float64)
comps = MultivariateNormal(means, (s ** 2) * torch.eye(2, dtype=torch.float64).expand(2, 2, 2))
mix = MixtureSameFamily(Categorical(torch.tensor([pi, 1 - pi], dtype=torch.float64)), comps)


def responsibilities(x):
    logw = torch.log(torch.tensor([pi, 1 - pi], dtype=torch.float64)) + comps.log_prob(x[:, None, :])
    return torch.softmax(logw, dim=1)


# ── Mixture score = responsibility-weighted average of component scores ─────
x = torch.tensor([[-2.0, 1.0], [0.0, 0.5], [0.3, -1.0], [2.5, 0.0]], dtype=torch.float64, requires_grad=True)
(auto,) = torch.autograd.grad(mix.log_prob(x).sum(), x)
r = responsibilities(x.detach())
comp_scores = -(x.detach()[:, None, :] - means) / s ** 2           # each component's own score
weighted = (r[:, :, None] * comp_scores).sum(1)
print("point          responsibilities       autograd score         weighted component scores")
for p_, rr, a, w in zip(x.tolist(), r.tolist(), auto.tolist(), weighted.tolist()):
    print(f"{str(p_):13s}  {str([round(v, 3) for v in rr]):20s}  {str([round(v, 3) for v in a]):21s}  {[round(v, 3) for v in w]}")
print("match:", torch.allclose(auto, weighted))

# ── Between the modes the field bends: at the midpoint the two pulls cancel ─
mid = torch.tensor([[0.0, 0.0]], dtype=torch.float64, requires_grad=True)
(sm,) = torch.autograd.grad(mix.log_prob(mid).sum(), mid)
print("\nscore at the midpoint (equal weights):", sm.tolist())

point          responsibilities       autograd score         weighted component scores
[-2.0, 1.0]    [1.0, 0.0]            [1.423, -1.778]        [1.423, -1.778]
[0.0, 0.5]     [0.5, 0.5]            [0.0, -0.889]          [0.0, -0.889]
[0.3, -1.0]    [0.218, 0.782]        [0.672, 1.778]         [0.672, 1.778]
[2.5, 0.0]     [0.0, 1.0]            [-2.311, 0.0]          [-2.311, 0.0]
match: True

score at the midpoint (equal weights): [[0.0, 0.0]]
